### XGBoost

In [1]:
import pandas as pd
from utils import *
from xgboost import XGBClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.calibration import CalibratedClassifierCV

In [2]:
df = pd.read_csv('../../../data/creditcard.csv')
df = create_features(df)
df['hour_sin'] = np.sin(2 * np.pi * df['Hour_from_start_mod24']/24)
df['hour_cos'] = np.cos(2 * np.pi * df['Hour_from_start_mod24']/24)
df['time_diff'] = df['Time'].diff().fillna(0)
threshold = df['Amount'].quantile(0.95)  
df['is_high_amount'] = (df['Amount'] > threshold).astype(int)
df['is_rapid_transaction'] = (df['time_diff'] < 60).astype(int)
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,Class,_log_amount,Hour_from_start_mod24,is_night_proxy,is_business_hours_proxy,hour_sin,hour_cos,time_diff,is_high_amount,is_rapid_transaction
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,0,5.014760,0,1,0,0.0,1.0,0.0,0,1
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,0,1.305626,0,1,0,0.0,1.0,0.0,0,1
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0,5.939276,0,1,0,0.0,1.0,1.0,1,1
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,0,4.824306,0,1,0,0.0,1.0,0.0,0,1
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,0,4.262539,0,1,0,0.0,1.0,1.0,0,1


In [3]:
features = df.drop(['Time','Class','Amount','Hour_from_start_mod24'], axis=1).columns.tolist()
target = "Class"
X_train, y_train, X_val, y_val, X_test, y_test = split_data(df, features, target)

X_train: (181584, 36) y_train: (181584,)
X_val: (45396, 36) y_val: (45396,)
X_test: (56746, 36) y_test: (56746,)
Fraud rate in train: 0.001910961318177813
Fraud rate in test: 0.0013040566735981391


In [4]:
pos, neg = int((y_train==1).sum()), int((y_train==0).sum())
xg_model = XGBClassifier(
    n_estimators=800,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.6,
    colsample_bytree=0.9,
    reg_lambda = 1.0,
    gamma=1.0,
    min_child_weight=4,
    tree_method = "hist",
    scale_pos_weight=neg/max(pos, 1),
    eval_metric='aucpr',
    random_state=SEED
)

xg_model.fit(X_train, y_train)

p_val_xg  = xg_model.predict_proba(X_val)[:, 1]

In [5]:
rs = []

rs.append({
    "model": "XGB val",
    **log_eval(y_val, p_val_xg)
})

val_df = pd.DataFrame(rs)
val_df

,model,threshold,Cost,ROC_AUC,PR_AUC,debiased_ece,adaptive_ece,Brier
0,XGB val,0.028,2105.0,0.982716,0.794442,0.000378,0.000325,0.000494


In [6]:
eval = []

for n in np.linspace(0,1,11):

    eval.append({
        "model": "XGB val",
        **evaluate(y_val, p_val_xg, thr = n) 
    })

eval_df = pd.DataFrame(eval)
eval_df

,model,threshold,precision,recall,f1,roc_auc,auprc,brier,tp,fp,fn,tn
0,XGB val,0.0,0.001145,1.000000,0.002288,0.982716,0.794442,0.000494,52,45344,0,0
1,XGB val,0.1,0.567568,0.807692,0.666667,0.982716,0.794442,0.000494,42,32,10,45312
2,XGB val,0.2,0.640625,0.788462,0.706897,0.982716,0.794442,0.000494,41,23,11,45321
3,XGB val,0.3,0.655738,0.769231,0.707965,0.982716,0.794442,0.000494,40,21,12,45323
4,XGB val,0.4,0.666667,0.769231,0.714286,0.982716,0.794442,0.000494,40,20,12,45324
5,XGB val,0.5,0.696429,0.750000,0.722222,0.982716,0.794442,0.000494,39,17,13,45327
6,XGB val,0.6,0.750000,0.750000,0.750000,0.982716,0.794442,0.000494,39,13,13,45331
7,XGB val,0.7,0.812500,0.750000,0.780000,0.982716,0.794442,0.000494,39,9,13,45335
8,XGB val,0.8,0.863636,0.730769,0.791667,0.982716,0.794442,0.000494,38,6,14,45338
9,XGB val,0.9,0.926829,0.730769,0.817204,0.982716,0.794442,0.000494,38,3,14,45341


In [7]:
p_test_xg = xg_model.predict_proba(X_test)[:,1]

In [8]:
rs.append({
    "model": "XGB test",
    **log_eval(y_test, p_test_xg)
})

val_df = pd.DataFrame(rs)
val_df

,model,threshold,Cost,ROC_AUC,PR_AUC,debiased_ece,adaptive_ece,Brier
0,XGB val,0.028,2105.0,0.982716,0.794442,0.000378,0.000325,0.000494
1,XGB test,0.027,3255.0,0.966922,0.795390,0.000360,0.000330,0.000484


In [9]:
result_cal = evaluate(y_test, p_test_xg, 0.7)
print(f"TP={result_cal['tp']}, FP={result_cal['fp']}, FN={result_cal['fn']}, TN={result_cal['tn']}")
print(pd.Series(y_test).value_counts())

TP=56, FP=8, FN=18, TN=56664
Class
0    56672
1       74
Name: count, dtype: int64


In [10]:
xgb_features = X_train.columns.tolist()
import joblib 
joblib.dump(xgb_features, "../../../artifacts/xgb_features.joblib")

['../../../artifacts/xgb_features.joblib']

In [11]:
joblib.dump(xg_model, "../../../artifacts/xgb_model.joblib")

['../../../artifacts/xgb_model.joblib']

### Caliration

In [12]:
tscv = TimeSeriesSplit(n_splits=5)

In [13]:
cal_xg = CalibratedClassifierCV(
    estimator=xg_model,
    method='isotonic',
    cv=tscv
)

cal_xg.fit(X_val, y_val)

p_test_cal_xg = cal_xg.predict_proba(X_test)[:, 1]

rs.append({
    "model": "Cal XGB",
    **log_eval(y_test, p_test_cal_xg)
})

val_df = pd.DataFrame(rs)
val_df

,model,threshold,Cost,ROC_AUC,PR_AUC,debiased_ece,adaptive_ece,Brier
0,XGB val,0.028,2105.0,0.982716,0.794442,0.000378,0.000325,0.000494
1,XGB test,0.027,3255.0,0.966922,0.795390,0.000360,0.000330,0.000484
2,Cal XGB,0.070,3685.0,0.938429,0.755640,0.000168,0.000129,0.000451


In [14]:
result_cal = evaluate(y_test, p_test_cal_xg)
print(f"TP={result_cal['tp']}, FP={result_cal['fp']}, FN={result_cal['fn']}, TN={result_cal['tn']}")
print(pd.Series(y_test).value_counts())

TP=52, FP=5, FN=22, TN=56667
Class
0    56672
1       74
Name: count, dtype: int64
